In [1]:
import os
from pathlib import Path
import numpy as np

import pandas as pd

In [2]:
project_root = Path(os.environ["PROJECT_ROOT"])

In [3]:
hcp_pheno = pd.read_csv(os.environ["HCP_PHENO_UNRESTRICTED"], dtype={"Subject": str})
hcp_pheno.set_index("Subject", inplace=True)
print(hcp_pheno.iloc[:, :4].head(5))

        Release Acquisition Gender    Age
Subject                                  
100004     S900         Q06      M  22-25
100206     S900         Q11      M  26-30
100307       Q1         Q01      F  26-30
100408       Q3         Q03      M  31-35
100610     S900         Q08      M  26-30


In [4]:
hcp_58_columns = (
    (project_root / "resources/column_lists/58behaviors_age_sex.txt")
    .read_text()
    .splitlines()
)[:58]

hcp_behav = hcp_pheno.loc[:, hcp_58_columns]

complete_behav_mask = ~hcp_behav.isna().any(axis=1)
print("Num with complete behavioral data:", complete_behav_mask.sum())

Num with complete behavioral data: 1022


In [5]:
complete_rest_mask = hcp_pheno["3T_RS-fMRI_Count"] == 4
print("Num with complete 3T rest data:", complete_rest_mask.sum())

Num with complete 3T rest data: 1018


In [6]:
hcp_fd_dir = project_root / "results/hcp_1200_rfmri_fd"
hcp_fd = pd.read_parquet(hcp_fd_dir / "hcp_1200_rfmri_fd.parquet")

# Only include 3T data
hcp_fd = hcp_fd.query("mag == '3T'")

# Aggregate over subject
hcp_fd = hcp_fd.groupby("sub").agg({"task": "count", "fd": "mean"})
hcp_fd.columns = ["run_count", "mean_fd"]
hcp_fd.index.name = "Subject"

print(hcp_fd.head(5))

         run_count   mean_fd
Subject                     
100206           4  0.108884
100307           4  0.125204
100408           4  0.183484
100610           4  0.180003
101006           4  0.155110


In [7]:
fd_threshold = 0.3
low_fd_mask = hcp_fd["mean_fd"] < fd_threshold
print(f"Num with mean FD < {fd_threshold}:", low_fd_mask.sum())

Num with mean FD < 0.3: 1055


In [8]:
hcp_include_mask = complete_behav_mask & complete_rest_mask & low_fd_mask
print("Total num include:", hcp_include_mask.sum())

Total num include: 955


In [9]:
rng = np.random.default_rng(7582)
hcp_include_subs = hcp_include_mask.index[hcp_include_mask].values
# Shuffle order so that each subsequence is a random sample
rng.shuffle(hcp_include_subs)
print(hcp_include_subs[:8])

['285446' '133827' '429040' '122317' '126628' '206929' '118023' '901038']


In [10]:
n_subs = len(hcp_include_subs)
np.savetxt(
    project_root
    / f"resources/subject_lists/hcp_complete_data_fd{fd_threshold}_{n_subs}.txt",
    hcp_include_subs,
    fmt="%s",
)